# Week 4 · Day 1 — NumPy Arrays + Central Tendency

**Course:** AI (Machine Learning & Deep Learning) — NAVTTC Skills for All
**Mode:** Live coding — type every line yourself, run it, and watch what happens.

### Today's map
| Hour | Topic |
|---|---|
| 1 | Why NumPy · creating arrays |
| 2 | Array attributes · reshaping |
| 3 | 📊 STATS: mean, median, mode |
| 4 | Copy vs view · array operations |

**End of day:** push this notebook to your GitHub repo as `week-04/day1-numpy-basics.ipynb`

> **INSTRUCTOR NOTES (delete before sharing):**
> - Keep the motivational-lecture opening to 10 min max or Hour 4 gets squeezed.
> - Rule of thumb all week: never talk more than 5 minutes without running code.
> - Trigger the errors marked ⚠️ *on purpose* and read the traceback aloud with the class — normalizing error messages is half the battle in Week 4.

---
## Hour 1 — Why NumPy?

Python lists are flexible but **slow** for math. NumPy arrays are fixed-type, stored in contiguous memory, and use fast C loops under the hood. Don't take my word for it — let's time it.

In [ ]:
# The opening hook: sum 10 million numbers, list vs array
big_list = list(range(10_000_000))

%timeit sum(big_list)

In [ ]:
import numpy as np

big_arr = np.arange(10_000_000)

%timeit big_arr.sum()

> **INSTRUCTOR:** Expect roughly 50–100× speedup. Ask the class: *"if one model training step takes 1 second with NumPy, how long with plain lists?"* Let them do the math — that's why every ML library is built on NumPy.

**Result:** typically 50–100× faster. Every library we use this course (Pandas, scikit-learn, PyTorch) is built on top of NumPy — this is why.

### Creating arrays — from Python lists

In [ ]:
my_list = [1, 2, 3, 4, 5]
arr = np.array(my_list)
arr

In [ ]:
# 2D: a list of lists becomes a matrix
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
matrix

In [ ]:
type(arr), type(matrix)   # same type! dimensions differ, class doesn't

### Creating arrays — built-in generators

In [ ]:
np.arange(0, 10)        # like range(), but returns an array

In [ ]:
np.arange(0, 51, 5)     # start, stop (exclusive), step

In [ ]:
np.linspace(0, 1, 11)   # 11 evenly spaced points from 0 to 1 INCLUSIVE
# arange = "step size", linspace = "how many points" 

In [ ]:
np.zeros(5), np.zeros((3, 4))   # a tuple gives you 2D

In [ ]:
np.ones((2, 3)) * 7      # quick way to fill with any value

In [ ]:
np.eye(4)                # identity matrix — you'll meet this again in linear algebra

### Creating arrays — random numbers

Randomness is everywhere in ML (weight initialization, shuffling, sampling). The **seed** makes randomness *reproducible* — same seed, same "random" numbers, on any machine.

In [ ]:
np.random.seed(42)       # everyone runs this → everyone gets identical results
np.random.rand(5)        # uniform [0, 1)

In [ ]:
np.random.randn(5)       # standard normal: mean 0, std 1 — remember this for Hour 3 of Day 2!

In [ ]:
np.random.randint(1, 101, 10)    # 10 random integers from 1 to 100

---
### 🎯 SOLO TASK 1 (10 minutes)

1. Create a **5×5 matrix** of random integers between 1 and 100.
2. Create a **1D array of every even number from 2 to 50** (inclusive).

Try it yourself before peeking at anyone's screen.

In [ ]:
# SOLUTION 1a
task1a = np.random.randint(1, 101, (5, 5))
task1a

In [ ]:
# SOLUTION 1b  — common mistake: stopping at 50 excludes it; must go to 51
task1b = np.arange(2, 51, 2)
task1b

---
## Hour 2 — Array Attributes & Reshaping

Attributes are *facts about* the array (no parentheses — they're not functions).

In [ ]:
arr = np.arange(0, 30)
arr.shape, arr.dtype, arr.size, arr.ndim

In [ ]:
matrix = np.random.randint(1, 100, (4, 5))
print("shape:", matrix.shape)   # (rows, columns)
print("dtype:", matrix.dtype)
print("size :", matrix.size)    # total number of elements
print("ndim :", matrix.ndim)    # number of dimensions

### Reshaping

`reshape` rearranges the same data into a new shape — **the total number of elements must not change**.

In [ ]:
arr = np.arange(0, 30)
arr.reshape(5, 6)

In [ ]:
arr.reshape(6, -1)   # -1 means "you figure out this dimension for me"
# 30 elements / 6 rows = 5 columns. You'll use -1 constantly in deep learning.

⚠️ **Let's break it on purpose.** What happens if the sizes don't match?

In [ ]:
# This SHOULD fail — read the error message carefully. It tells you exactly what's wrong.
arr.reshape(4, 7)   # 4 x 7 = 28 ≠ 30

> **INSTRUCTOR:** Read the ValueError aloud: *"cannot reshape array of size 30 into shape (4,7)"*. Point out that NumPy errors usually say precisely what's wrong — students who read errors debug 10× faster than students who panic.

### Max, min — and *where* they are

In [ ]:
np.random.seed(7)
scores = np.random.randint(0, 100, 10)
scores

In [ ]:
scores.max(), scores.min()

In [ ]:
scores.argmax(), scores.argmin()   # the INDEX (position) of max / min
# arg* functions matter later: "which class did the model predict?" = argmax of probabilities

---
### 🎯 SOLO TASK 2 (10 minutes)

Take `np.arange(1, 31)`, reshape it to **5×6**, then find the **maximum value and its position**.

In [ ]:
# SOLUTION 2
task2 = np.arange(1, 31).reshape(5, 6)
print(task2)
print("max value:", task2.max())
print("flat position:", task2.argmax())
# Bonus for a sharp student: np.unravel_index(task2.argmax(), task2.shape) → (row, col)

---
## Hour 3 — 📊 Measures of Central Tendency

*One number that represents the whole dataset.* Three candidates:

| Measure | Definition | Sensitive to outliers? |
|---|---|---|
| **Mean** | sum ÷ count | **Yes — badly** |
| **Median** | middle value when sorted | No |
| **Mode** | most frequent value | No |

**The billionaire question:** a billionaire walks into this classroom. What happens to the *mean* income of the room? What happens to the *median*? Keep that in mind — we'll prove it in code in a few minutes.

> **INSTRUCTOR:** Whiteboard 5 min max — one 7-number example, compute all three by hand, ask the billionaire question, then straight back to the notebook. Don't define skewness yet; that word arrives on Day 5 when they can *see* it.

In [ ]:
salaries = np.array([55, 60, 45, 70, 65, 50, 58, 62, 48, 66])   # in thousands PKR/month
salaries

In [ ]:
# Mean, computed MANUALLY first — no black boxes in this course
salaries.sum() / salaries.size

In [ ]:
# ...and now the built-in
salaries.mean(), np.median(salaries)

In [ ]:
# Mode: NumPy doesn't have one! The scientific Python ecosystem is bigger than one library.
from scipy import stats
grades = np.array([3, 7, 7, 2, 7, 5, 3, 7])
stats.mode(grades)

### The outlier experiment — proving the billionaire question

In [ ]:
# The billionaire (in thousands: 10 crore/month) joins our salary data
with_outlier = np.append(salaries, 100_000)

print("BEFORE  → mean:", salaries.mean(), "   median:", np.median(salaries))
print("AFTER   → mean:", round(with_outlier.mean(), 1), " median:", np.median(with_outlier))

**One extreme value dragged the mean from ~58 to ~9,100 — while the median barely moved.**

This is why news reports use *median* household income, and why we'll fill missing values with the *median* on Day 4. The mean is honest only when the data has no extreme values.

---
### 🎯 SOLO TASK 3 (15 minutes)

Below are 20 monthly salaries (thousands PKR) from a small company — including the CEO.

1. Compute the **mean** and the **median**.
2. In a one-line comment, state **which one better represents a typical employee's salary, and why**.

In [ ]:
company = np.array([42, 38, 45, 50, 41, 39, 47, 44, 52, 40,
                    43, 46, 38, 49, 41, 45, 39, 48, 44, 950])  # last one = CEO

# SOLUTION 3
mean_salary = company.mean()          # ≈ 89.05 — higher than 19 of the 20 employees!
median_salary = np.median(company)    # 44.0
print(mean_salary, median_salary)

# ANSWER: the median. The CEO's 950 is an outlier that drags the mean up to ~89,
# a figure that describes almost nobody. The median (44) sits where typical salaries actually are.

---
## Hour 4 — Copy vs View, and Array Operations

### ⚠️ The aliasing bug — the most common NumPy mistake

Slicing an array does **not** copy the data. A slice is a *view* — a window onto the same memory.

In [ ]:
original = np.arange(10)
window = original[0:5]      # looks like a new array...
window[:] = 999             # ...modify the "copy"
original                    # 😱 the original changed too!

In [ ]:
# The fix: .copy() when you truly want independent data
original = np.arange(10)
safe = original[0:5].copy()
safe[:] = 999
print("safe    :", safe)
print("original:", original)   # untouched ✅

> **INSTRUCTOR:** Let the 😱 moment happen — pause after `original` prints and wait for the reaction before explaining. Views exist for speed (no copying 10M elements just to look at half of them). They will hit this bug in Pandas too (`SettingWithCopyWarning`) — plant that flag now.

### Adding, removing, sorting

In [ ]:
arr = np.array([3, 1, 4, 1, 5])

np.append(arr, [9, 2])        # note: returns a NEW array — arr is unchanged

In [ ]:
np.insert(arr, 1, 100)        # insert 100 at index 1

In [ ]:
np.delete(arr, 0)             # remove element at index 0

In [ ]:
np.sort(arr)                  # returns a sorted COPY (arr.sort() would sort in place)

### Combining and splitting

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
np.concatenate([a, b])

In [ ]:
m1 = np.ones((2, 3))
m2 = np.zeros((2, 3))
print(np.concatenate([m1, m2], axis=0))   # stack as rows (on top of each other)
print(np.concatenate([m1, m2], axis=1))   # stack as columns (side by side)
# axis will haunt you all course — axis=0 is "down the rows", axis=1 is "across the columns" 

In [ ]:
big = np.arange(12)
np.split(big, 3)              # 3 equal pieces

---
## ✅ End of Day 1

**Before you leave:**
1. Restart & Run All (`Kernel → Restart & Run All`) — every cell must run top-to-bottom without errors (the deliberate ⚠️ reshape error cell is the one exception — comment it out first).
2. Push to GitHub: `week-04/day1-numpy-basics.ipynb` in your `ai-course-journal` repo.

**Homework (reinforcement):** W3Schools NumPy exercises — original syllabus **Tasks 28–35** (creating arrays, indexing, slicing, data types, copy vs view, shape, reshape, iteration).

**Tomorrow:** indexing & boolean selection (the backbone of everything in Pandas), broadcasting — and your first z-scores. Bring today's mean/median instincts with you.